In [1]:
import leafmap.maplibregl as leafmap

import os
os.environ["MAPTILER_KEY"] = "G7M6sxUdhhwyWa0BLf5W"

import ee
ee.Authenticate()
ee.Initialize(project='wired-day-365517')

import geemap

import pandas as pd

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_7TDKVSyKvBdmMqW?ref=4i2o6


In [2]:
NFC_FACILITIES_PATH = '/usr2/people/macgregor/amplicon/test/data/nfc/facilities.tsv'
def clean_df(df):
    df = df.dropna(subset=['latitude', 'longitude'])
    df = df[df['latitude'].between(-90, 90)]
    df = df[df['longitude'].between(-180, 180)]
    return df
nfc_df = pd.read_csv(NFC_FACILITIES_PATH, sep='\t', low_memory=False)
nfc_df = clean_df(nfc_df)
nfc_df = nfc_df.astype(object).where(pd.notna(nfc_df), None)

nfc_points_ee = geemap.pandas_to_ee(
    nfc_df,
    latitude_col='latitude',
    longitude_col='longitude'
)

In [4]:
SAMPLES_PATH = '/usr2/people/macgregor/amplicon/test/data/merged/metadata/final_metadata.tsv'
# 1. Load the data
samples_df = pd.read_csv(SAMPLES_PATH, sep='\t', low_memory=False)
print(f"Initial shape: {samples_df.shape}")

# 2. First, handle NaN-to-None conversion for all OTHER property columns.
samples_df = samples_df.astype(object).where(pd.notna(samples_df), None)

# 3. NOW, perform the strict cleaning on the coordinate columns as the FINAL step.
samples_df['latitude_deg'] = pd.to_numeric(samples_df['latitude_deg'], errors='coerce')
samples_df['longitude_deg'] = pd.to_numeric(samples_df['longitude_deg'], errors='coerce')
samples_df.dropna(subset=['latitude_deg', 'longitude_deg'], inplace=True)
samples_df = samples_df[samples_df['latitude_deg'].between(-90, 90)]
samples_df = samples_df[samples_df['longitude_deg'].between(-180, 180)]
# 1. Reset the index. This creates a new, sequential index.
print("Resetting the index...")
samples_df.reset_index(inplace=True)
samples_df.drop(columns=['latitude', 'longitude'], inplace=True)
samples_df.rename(
    columns={'latitude_deg': 'latitude', 'longitude_deg': 'longitude'},
    inplace=True
)
samples_df = samples_df[['latitude', 'longitude', '#sampleid', 'dataset_name', 'collection_date', 'nuclear_contamination_status', 'facility_match']]
# 2. Immediately verify the result.
print("\n--- Verifying data AFTER resetting index ---")
samples_df.info()

# 4. Convert to an Earth Engine object. This should now be safe.
if not samples_df.empty:
    sample_points_ee = geemap.pandas_to_ee(
        samples_df,
        latitude_col='latitude',
        longitude_col='longitude'
    )
    print("\n✅ Successfully converted DataFrame to ee.FeatureCollection.")
else:
    print("\n❌ DataFrame is empty after cleaning.")

Initial shape: (16766, 102)
Resetting the index...

--- Verifying data AFTER resetting index ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14511 entries, 0 to 14510
Data columns (total 7 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   latitude                      14511 non-null  float64
 1   longitude                     14511 non-null  float64
 2   #sampleid                     14511 non-null  object 
 3   dataset_name                  14244 non-null  object 
 4   collection_date               11212 non-null  object 
 5   nuclear_contamination_status  4911 non-null   object 
 6   facility_match                14511 non-null  object 
dtypes: float64(2), object(5)
memory usage: 793.7+ KB

✅ Successfully converted DataFrame to ee.FeatureCollection.


In [6]:
# Create a map centered on the US
Map = geemap.Map(center=[40, -98], zoom=4)

# --- Layer 1: Soil Organic Carbon ---
soc_image = ee.Image('projects/soilgrids-isric/soc_mean').select('soc_0-5cm_mean')
soc_vis = {
    'min': 0,
    'max': 50,  # Units are decigrams per kilogram
    'palette': ['#ffffd9', '#edf8b1', '#c7e9b4', '#7fcdbb', '#41b6c4', '#1d91c0', '#225ea8']
}
Map.addLayer(soc_image, soc_vis, 'Soil Organic Carbon (0-5cm)')

# --- Layer 2: Clay Content ---
clay_image = ee.Image('projects/soilgrids-isric/clay_mean').select('clay_0-5cm_mean')
clay_vis = {
    'min': 0,
    'max': 60,  # Units are grams per 100g (percent)
    'palette': ['#f7fcf5', '#e5f5e0', '#c7e9c0', '#a1d99b', '#74c476', '#41ab5d', '#238b45']
}
Map.addLayer(clay_image, clay_vis, 'Clay Content (0-5cm)', shown=False)

# --- Layer 3: Soil pH ---
ph_image = ee.Image('projects/soilgrids-isric/phh2o_mean').select('phh2o_0-5cm_mean')
ph_vis = {
    'min': 40,
    'max': 120,  # pH value multiplied by 10 (e.g., 70 = pH 7.0)
    'palette': ['#d7191c', '#fdae61', '#ffffbf', '#abdda4', '#2b83ba'] # Red=Acidic, Blue=Alkaline
}
Map.addLayer(ph_image, ph_vis, 'Soil pH (0-5cm)', shown=False)

# --- Add your facility points on top ---
point_style = {'color': '#000000', 'pointSize': 3}
Map.addLayer(nfc_points_ee, point_style, 'NFC Facilities')
# --- Add your facility points on top ---
point_style = {'color': "#A32525", 'pointSize': 3}
Map.addLayer(sample_points_ee, point_style, 'ENA Samples')


# Add the layer control to toggle layers on and off
Map.addLayerControl()

# Display the map
Map

Map(center=[40, -98], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', tra…

In [ ]:
m = leafmap.Map(
    center=[-120.4482, 38.0399], zoom=13, pitch=60, bearing=30, style="3d-terrain"
)
m.add_ee_layer(asset_id="ESA/WorldCover/v200", opacity=0.5)
m.add_legend(builtin_legend="ESA_WorldCover", title="ESA Landcover")
m.add_layer_control()
m

In [ ]:
m.layer_interact()

In [ ]:
m = leafmap.Map(
    center=[-74.012998, 40.70414], zoom=15.6, pitch=60, bearing=30, style="3d-terrain"
)
m.add_ee_layer(asset_id="ESA/WorldCover/v200", opacity=0.5)
m.add_legend(builtin_legend="ESA_WorldCover", title="ESA Landcover")
m.add_layer_control()
MAPTILER_KEY = leafmap.get_api_key("MAPTILER_KEY")
source = {
    "url": f"https://api.maptiler.com/tiles/v3/tiles.json?key={MAPTILER_KEY}",
    "type": "vector",
}

layer = {
    "id": "3d-buildings",
    "source": "openmaptiles",
    "source-layer": "building",
    "type": "fill-extrusion",
    "min-zoom": 15,
    "paint": {
        "fill-extrusion-color": [
            "interpolate",
            ["linear"],
            ["get", "render_height"],
            0,
            "lightgray",
            200,
            "royalblue",
            400,
            "lightblue",
        ],
        "fill-extrusion-height": [
            "interpolate",
            ["linear"],
            ["zoom"],
            15,
            0,
            16,
            ["get", "render_height"],
        ],
        "fill-extrusion-base": [
            "case",
            [">=", ["get", "zoom"], 16],
            ["get", "render_min_height"],
            0,
        ],
    },
}
m.add_source("openmaptiles", source)
m.add_layer(layer)
m

In [ ]:
m = leafmap.Map(center=[-120.4482, 38.03994], zoom=13, pitch=60, bearing=30, style="3d-terrain")
dataset = ee.Image('USGS/SRTMGL1_003')
vis_params = {"bands": ["elevation"]}
m.add_ee_layer(dataset, vis_params, name="ESA Worldcover", opacity=0.5)
m.add_legend(builtin_legend="ESA_WorldCover", title="ESA Landcover")
m.add_layer_control()
m

In [ ]:
landsat7 = ee.Image("LANDSAT/LE7_TOA_5YEAR/1999_2003").select([0, 1, 2, 3, 4, 6])
landsat_vis = {"bands": ["B4", "B3", "B2"], "gamma": 1.4}
Map.addLayer(landsat7, landsat_vis, "LE7_TOA_5YEAR/1999_2003")

hyperion = ee.ImageCollection("EO1/HYPERION").filter(
    ee.Filter.date("2016-01-01", "2017-03-01")
)
hyperion_vis = {
    "min": 1000.0,
    "max": 14000.0,
    "gamma": 2.5,
}
Map.addLayer(hyperion, hyperion_vis, "EO1/HYPERION");

In [ ]:
Map.set_plot_options(plot_type="bar", add_marker_cluster=True)

In [ ]:
import ee
import geemap

# Authenticate and initialize
try:
    ee.Initialize(project='wired-day-365517')
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project='wired-day-365517')

# Assuming 'nfc_points_ee' is your FeatureCollection with 863 facilities.
# If not, you would load or create it here first.

# 1. Load the VIIRS Nighttime Lights data.
# We'll filter it for the year 2024 and calculate the average light intensity.
night_lights = (
    ee.ImageCollection('NOAA/VIIRS/DNB/MONTHLY_V1/VCMCFG')
    .filter(ee.Filter.date('2024-01-01', '2024-12-31'))
    .mean() 
    .select('avg_rad') # Create a single image representing the annual mean
)

# 2. Define the visualization parameters for the lights.
# We'll use a palette that goes from black (no light) to yellow (bright light).
night_lights_vis = {
    'min': 0.0,
    'max': 60.0, # Values over 60 are typically city centers
    'palette': ['#000000', '#323232', '#ffff00', '#ffffff']
}

# 3. Create a map and add the layers.
# The order you add them matters: layers added later appear on top.
Map = geemap.Map(center=[20, 0], zoom=2)

# Add the global nighttime image first (as the base layer)
Map.addLayer(night_lights, night_lights_vis, 'Nighttime Lights (2024)')


# Define a style for your facility points
point_style = {'color': '#FF5733', 'pointSize': 3} # Bright orange points

# Add your points on top of the lights layer
Map.addLayer(nfc_points_ee, point_style, 'NFC Facilities')

# Add a layer control panel to toggle layers on and off
Map.addLayerControl()

# Display the map
Map

In [ ]:
# 1. Define a point of interest (Berkeley) and a date range.
berkeley_point = ee.Geometry.Point([-122.2727, 37.8716])
# Search for images from this past summer.
start_date = '2025-06-01'
end_date = '2025-09-22'

# 2. Search for the best available Sentinel-2 image.
image = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    #.filterBounds(berkeley_point)
    #.filterDate(start_date, end_date)
    #.sort('CLOUDY_PIXEL_PERCENTAGE') # Sort by cloud cover, lowest first
    .first() # Get the single best image from the collection
)
# Define a style for the points.
point_style = {
    'color': '#0000FF', # Blue
    'pointSize': 5,
    'fillColor': '#0000FF88' # Semi-transparent blue fill
}
# Add the points to the map.
Map.addLayer(nfc_points_ee, point_style, 'NFC Facilities')
# 3. Define visualization parameters for a true-color composite.
vis_params = {
  'bands': ['B4', 'B3', 'B2'], # Red, Green, Blue bands
  'min': 0,
  'max': 3000,
  'gamma': 1.4,
}

# 4. Create a map and add the image to it.
Map = geemap.Map()
Map.addLayer(image, vis_params, 'Recent Sentinel-2 Image')
Map.centerObject(image, 11)

# Display the map.
Map

In [ ]:
# --- Step 2: Convert DataFrame to an ee.FeatureCollection ---



# --- Step 3: Style and Add the Points to a Map ---
Map = geemap.Map()
Map.addLayer(image, vis_params, 'Recent Sentinel-2 Image')
Map.centerObject(image, 11)

# Define a style for the points.
point_style = {
    'color': '#0000FF', # Blue
    'pointSize': 5,
    'fillColor': '#0000FF88' # Semi-transparent blue fill
}

# Add the points to the map.
Map.addLayer(nfc_points_ee, point_style, 'NFC Facilities')


# Add a basemap and display
Map.add_basemap('SATELLITE')
Map

In [ ]:
nfc_points_ee

In [ ]:
# Assuming 'nfc_points_ee' is the FeatureCollection from your previous step.

# 1. Get a boundary for California from a public dataset.
states = ee.FeatureCollection('TIGER/2018/States')
california_boundary = states.filter(ee.Filter.eq('NAME', 'California'))

# 2. Filter your 863 points to include only those within the California boundary.
california_points = nfc_points_ee.filterBounds(california_boundary)

# 3. Create a map and add the filtered points.
Map = geemap.Map()
Map.centerObject(california_boundary, 6) # Center the map on California

# Define styles
point_style = {'color': '#FF5733', 'pointSize': 5}
boundary_style = {'color': '#0000FF', 'fillColor': '#0000FF10'}

# Add layers to the map
Map.addLayer(california_boundary, boundary_style, 'California Boundary')
Map.addLayer(california_points, point_style, 'Facilities in California')
.first()
# You can check how many points were found
print('Number of facilities found in California:')
print(california_points.size().getInfo())

Map